# 9. Pose Transformer Counting Colab


## Reason, Approach, Result Interpretation

This stage checks two things at once:

- whether the existing pose-sequence augmentation path is worth keeping for a transformer encoder too
- whether a transformer over normalized pose tokens can improve on the shared pose TCN baselines for `squat`, `pull_up`, and `push_up`

This stage intentionally reuses the Stage 6 artifact contract so the resulting runs can be compared directly with the saved pose TCN baselines and the trivial train-split baseline.

Important caution:
- for `squat`, the real control is still the frozen dedicated squat run `squat_tcn_l1_channels96`
- this stage tests a transformer on the generic Stage 5 pose sequences, not on the dedicated squat-engineered feature branch


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TCN_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
TRANSFORMER_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_transformer.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')


def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [TCN_TRAIN_REL, TRANSFORMER_TRAIN_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'
TRAINING_OUTPUTS = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs'

print('POSE_INDEX =', POSE_INDEX)
print('Transformer trainer exists =', (DRIVE_PROJECT_ROOT / TRANSFORMER_TRAIN_REL).exists())


## Controlled Subset And Transformer Preset


In [ ]:
import pandas as pd

TRANSFORMER_RUNS = [
    {
        'exercise': 'squat',
        'seq_len': 256,
        'transformer_run': 'pose_transformer_squat_seq256',
        'pose_run': 'pose_count_tcn_squat_seq256',
        'pose_best_run': 'squat_tcn_l1_channels96',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'transformer_run': 'pose_transformer_pull_up_seq192',
        'pose_run': 'pose_count_tcn_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'transformer_run': 'pose_transformer_push_up_seq128',
        'pose_run': 'pose_count_tcn_push_up_seq128',
    },
]

AUGMENTATION = {
    'time_warp_range': '0.12',
    'feature_noise_std': '0.02',
    'frame_dropout_prob': '0.03',
    'camera_motion_std': '0.03',
    'camera_zoom_std': '0.05',
    'joint_occlusion_prob': '0.35',
    'joint_occlusion_min_ratio': '0.10',
    'joint_occlusion_max_ratio': '0.30',
    'joint_occlusion_max_joints': '2',
}

MODEL_DIM = 192
NUM_HEADS = 6
NUM_LAYERS = 4
FF_DIM = 384
DROPOUT = 0.20

meta_df = pd.read_csv(POSE_INDEX)
subset_df = meta_df[meta_df['type'].isin([cfg['exercise'] for cfg in TRANSFORMER_RUNS])]
display(subset_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index())
print('AUGMENTATION =', AUGMENTATION)
print('Transformer =', {
    'model_dim': MODEL_DIM,
    'num_heads': NUM_HEADS,
    'num_layers': NUM_LAYERS,
    'ff_dim': FF_DIM,
    'dropout': DROPOUT,
})


## Train Pose Transformer Runs


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in TRANSFORMER_RUNS:
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / TRANSFORMER_TRAIN_REL),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(POSE_INDEX),
        '--run-name', cfg['transformer_run'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', '80',
        '--batch-size', '16',
        '--lr', '0.0005',
        '--weight-decay', '0.0001',
        '--model-dim', str(MODEL_DIM),
        '--num-heads', str(NUM_HEADS),
        '--num-layers', str(NUM_LAYERS),
        '--ff-dim', str(FF_DIM),
        '--dropout', str(DROPOUT),
        '--patience', '15',
        '--loss', 'l1',
        '--eval-transform', 'raw',
        '--selection-metric', 'mae',
        '--sampler', 'balanced_count',
        '--time-warp-range', AUGMENTATION['time_warp_range'],
        '--feature-noise-std', AUGMENTATION['feature_noise_std'],
        '--frame-dropout-prob', AUGMENTATION['frame_dropout_prob'],
        '--camera-motion-std', AUGMENTATION['camera_motion_std'],
        '--camera-zoom-std', AUGMENTATION['camera_zoom_std'],
        '--joint-occlusion-prob', AUGMENTATION['joint_occlusion_prob'],
        '--joint-occlusion-min-ratio', AUGMENTATION['joint_occlusion_min_ratio'],
        '--joint-occlusion-max-ratio', AUGMENTATION['joint_occlusion_max_ratio'],
        '--joint-occlusion-max-joints', AUGMENTATION['joint_occlusion_max_joints'],
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['transformer_run'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All pose-transformer runs completed.')


## Pose TCN vs Pose Transformer Review


In [ ]:
import json
import pandas as pd

rows = []
for cfg in TRANSFORMER_RUNS:
    variants = [
        ('pose_tcn', cfg['pose_run']),
        ('pose_transformer', cfg['transformer_run']),
    ]
    if cfg.get('pose_best_run'):
        variants.insert(1, ('pose_best_squat', cfg['pose_best_run']))
    for variant, run_name in variants:
        metrics_path = TRAINING_OUTPUTS / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(['exercise', 'variant']))

pivot_df = compare_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['valid_mae', 'valid_within_1'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'valid_mae_pose_transformer' in pivot_df.columns and 'valid_mae_pose_tcn' in pivot_df.columns:
    pivot_df['delta_mae_transformer_minus_tcn'] = pivot_df['valid_mae_pose_transformer'] - pivot_df['valid_mae_pose_tcn']
if 'valid_within_1_pose_transformer' in pivot_df.columns and 'valid_within_1_pose_tcn' in pivot_df.columns:
    pivot_df['delta_within_1_transformer_minus_tcn'] = pivot_df['valid_within_1_pose_transformer'] - pivot_df['valid_within_1_pose_tcn']
if 'valid_mae_pose_best_squat' in pivot_df.columns and 'valid_mae_pose_transformer' in pivot_df.columns:
    pivot_df['delta_mae_transformer_minus_best_squat'] = pivot_df['valid_mae_pose_transformer'] - pivot_df['valid_mae_pose_best_squat']
if 'valid_within_1_pose_best_squat' in pivot_df.columns and 'valid_within_1_pose_transformer' in pivot_df.columns:
    pivot_df['delta_within_1_transformer_minus_best_squat'] = pivot_df['valid_within_1_pose_transformer'] - pivot_df['valid_within_1_pose_best_squat']
display(pivot_df.sort_values('exercise'))


## Compare Each Transformer Run Against The Trivial Baseline


In [ ]:
import json
import subprocess
import pandas as pd

comparison_failures = []
comparison_rows = []
for cfg in TRANSFORMER_RUNS:
    run_dir = TRAINING_OUTPUTS / cfg['transformer_run']
    predictions_csv = run_dir / 'predictions.csv'
    summary_json = run_dir / 'baseline_comparison_summary.json'
    rows_csv = run_dir / 'baseline_comparison_rows.csv'
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
        '--index-csv', str(POSE_INDEX),
        '--predictions-csv', str(predictions_csv),
        '--exercise', cfg['exercise'],
        '--output-json', str(summary_json),
        '--output-csv', str(rows_csv),
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['transformer_run'],
            'returncode': exc.returncode,
        })
        continue

    with open(summary_json, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    comparison_rows.append({
        'exercise': cfg['exercise'],
        'run_name': cfg['transformer_run'],
        'model_mae': summary['model_metrics']['mae'],
        'baseline_mae': summary['baseline_metrics']['mae'],
        'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
        'model_within_1': summary['model_metrics']['within_1'],
        'baseline_within_1': summary['baseline_metrics']['within_1'],
        'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
        'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
        'valid_rows': summary['row_level']['valid_rows'],
    })

if comparison_failures:
    display(pd.DataFrame(comparison_failures))

display(pd.DataFrame(comparison_rows).sort_values('exercise'))
